# NYC Taxi — Write Load Test
Reads the NYC taxi delta table, applies a handful of lightweight transformations, then **cross-joins with a multiplier range** to inflate the dataset to hundreds of millions of rows. The resulting giant table is written to a new delta table — no partition tricks, just raw volume.

> **Expected write time: 5–20 minutes** depending on cluster size and chosen multiplier.

In [ ]:
import time
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

start_time = time.time()

SOURCE_TABLE = "nyc_taxi"          # adjust to match your table name
TARGET_TABLE = "nyc_taxi_load_test_result"

# How many times to replicate the source dataset.
# 2 GB source × 50 = ~100 GB written to the new table.
# Increase to 100 for a longer run.
MULTIPLIER = 50

print(f"Source : {SOURCE_TABLE}")
print(f"Target : {TARGET_TABLE}")
print(f"Multiplier: {MULTIPLIER}x")
print(f"Started at: {time.strftime('%H:%M:%S')}")

## Step 1 — Load & lightweight transforms
Read the source delta table and add a small set of derived columns. Nothing expensive here.

In [ ]:
df = spark.read.format("delta").load(f"Tables/{SOURCE_TABLE}")

df_transformed = (
    df
    .withColumn("pickup_date",   F.to_date("tpep_pickup_datetime"))
    .withColumn("pickup_hour",   F.hour("tpep_pickup_datetime"))
    .withColumn("day_of_week",   F.dayofweek("tpep_pickup_datetime"))
    .withColumn("trip_duration_min",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60.0
    )
    .withColumn("speed_mph",
        F.when(
            (F.col("trip_distance") > 0) & (F.col("trip_duration_min") > 0),
            F.col("trip_distance") / (F.col("trip_duration_min") / 60.0)
        ).otherwise(F.lit(None).cast(DoubleType()))
    )
    .withColumn("cost_per_mile",
        F.when(F.col("trip_distance") > 0, F.col("total_amount") / F.col("trip_distance"))
         .otherwise(F.lit(None).cast(DoubleType()))
    )
    .withColumn("tip_rate",
        F.when(F.col("fare_amount") > 0, F.col("tip_amount") / F.col("fare_amount"))
         .otherwise(F.lit(0.0))
    )
    .withColumn("fare_bucket",
        F.when(F.col("fare_amount") <  5,  "micro")
         .when(F.col("fare_amount") < 15,  "low")
         .when(F.col("fare_amount") < 30,  "medium")
         .when(F.col("fare_amount") < 60,  "high")
         .otherwise("premium")
    )
)

source_count = df_transformed.count()
print(f"Source rows  : {source_count:,}")
print(f"Elapsed      : {(time.time()-start_time)/60:.1f} min")

## Step 2 — Blow up the data
Cross-join with `spark.range(MULTIPLIER)` to replicate every row N times.
A 2 GB source at 50× produces ~100 GB of data that Spark must write in full.

In [ ]:
# Cross-join with a tiny range DataFrame — no shuffle, pure row explosion
multiplier_df = spark.range(MULTIPLIER).withColumnRenamed("id", "replica_id")

df_blown = df_transformed.crossJoin(multiplier_df)

blown_count = df_blown.count()
print(f"Blown-up rows: {blown_count:,}  ({blown_count / source_count:.0f}× source)")
print(f"Elapsed      : {(time.time()-start_time)/60:.1f} min")

## Step 3 — Write to new delta table
Plain `overwrite` write of the full blown-up dataset. No tricks — the slowness comes entirely from the volume of data being written.

In [ ]:
write_start = time.time()

df_blown.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"Tables/{TARGET_TABLE}")

write_min = (time.time() - write_start) / 60
total_min = (time.time() - start_time) / 60

print(f"Write done!")
print(f"  Rows written : {blown_count:,}")
print(f"  Write time   : {write_min:.1f} min")
print(f"  Total time   : {total_min:.1f} min")
print(f"  Target table : {TARGET_TABLE}")